# Claim Documents: Profile Both Approaches

Pulls results from both runs and produces a side-by-side comparison:

| | `ai_parse_document` -> `ai_query` | FMAPI DocumentContent + Ray |
|-|-|-|
| accuracy | per-file, per-field | per-file, per-field |
| throughput | wall_s, docs/s | wall_s, docs/s |
| latency | n/a (SQL batch) | per-doc p50/p95 |
| path | SQL-native, governed | Python, direct to model |

**Prereqs.** Run [`claim_doc_ai_parse.ipynb`](claim_doc_ai_parse.ipynb) and
[`claim_doc_ray_claude.ipynb`](claim_doc_ray_claude.ipynb) first. They write
`shm.genai.benchmark_results` and `shm.genai.fmapi_ray_benchmark` respectively, plus
MLflow runs `aiquery_chain` and `fmapi_ray` under `document_intelligence_eval`.

In [ ]:
%pip install --quiet mlflow pandas
%restart_python

In [ ]:
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient

CATALOG, SCHEMA = "shm", "genai"
user = spark.sql("SELECT current_user()").first()[0]
EXPERIMENT = f"/Users/{user}/document_intelligence_eval"
mlflow.set_experiment(EXPERIMENT)

EVAL_FIELDS = ["document_type", "insurer_name", "insured_name", "policy_number",
               "effective_date", "expiration_date", "coverage_types"]

## 1. Per-file accuracy: do the two approaches agree?

Join on filename. A doc where the two approaches disagree (one right, one wrong) is
where each path's weakness lives — worth eyeballing the actual extractions.

In [ ]:
chain_df = (spark.table(f"{CATALOG}.{SCHEMA}.benchmark_results")
    .toPandas()
    .rename(columns={"accuracy": "acc_chain",
                     **{f: f"{f}_chain" for f in EVAL_FIELDS}}))
ray_df   = (spark.table(f"{CATALOG}.{SCHEMA}.fmapi_ray_benchmark")
    .toPandas()
    .rename(columns={"accuracy": "acc_ray", "latency_s": "ray_latency_s",
                     **{f: f"{f}_ray" for f in EVAL_FIELDS}}))

merged = chain_df.merge(ray_df, on="filename", how="outer")
merged["delta_accuracy"] = (merged["acc_ray"] - merged["acc_chain"]).round(3)
merged = merged.sort_values("delta_accuracy")

display(merged[["filename", "acc_chain", "acc_ray", "delta_accuracy", "ray_latency_s"]])

## 2. Aggregate summary — accuracy, wall time, throughput

Pulls wall time + throughput from the latest MLflow run of each approach so the
comparison is always grounded in the most recent execution, not a stale constant.

In [ ]:
client = MlflowClient()
exp = mlflow.get_experiment_by_name(EXPERIMENT)


def latest_metrics(run_name: str) -> dict:
    runs = client.search_runs(
        [exp.experiment_id],
        filter_string=f"tags.mlflow.runName = '{run_name}'",
        order_by=["attributes.start_time DESC"],
        max_results=1,
    )
    if not runs:
        return {}
    return runs[0].data.metrics


m_chain = latest_metrics("aiquery_chain")
m_ray   = latest_metrics("fmapi_ray")

summary = pd.DataFrame({
    "approach":        ["ai_parse -> ai_query", "FMAPI + Ray"],
    "mean_accuracy":   [m_chain.get("mean_accuracy"), m_ray.get("mean_accuracy")],
    "wall_s":          [m_chain.get("total_wall_s"),  m_ray.get("wall_s")],
    "docs_per_s":      [m_chain.get("docs_per_s"),    m_ray.get("docs_per_s")],
    "p50_latency_s":   [None,                         m_ray.get("p50_latency_s")],
    "p95_latency_s":   [None,                         m_ray.get("p95_latency_s")],
}).round(3)

display(summary)

## 3. Per-field accuracy breakdown

Which fields each approach wins on. Policy numbers and dates tend to be where the
vision-direct path shines (layout context); document type tends to be a wash.

In [ ]:
rows = []
for f in EVAL_FIELDS:
    chain_col = f"{f}_chain"
    ray_col   = f"{f}_ray"
    rows.append({
        "field":         f,
        "chain_mean":    merged[chain_col].mean() if chain_col in merged else None,
        "ray_mean":      merged[ray_col].mean()   if ray_col   in merged else None,
    })
per_field = pd.DataFrame(rows)
per_field["delta"] = (per_field["ray_mean"] - per_field["chain_mean"]).round(3)
per_field = per_field.round(3).sort_values("delta")

display(per_field)

## 4. Log the comparison as its own MLflow run

So the relative profile is versioned, not just the two runs it rolled up.

In [ ]:
with mlflow.start_run(run_name="profile_comparison"):
    mlflow.log_table(summary,   "summary.json")
    mlflow.log_table(per_field, "per_field.json")
    mlflow.log_table(merged,    "per_file.json")
    for col, key in [("mean_accuracy", "mean_accuracy"),
                     ("wall_s", "wall_s"),
                     ("docs_per_s", "docs_per_s")]:
        for i, approach in enumerate(["chain", "ray"]):
            val = summary.iloc[i][col]
            if pd.notna(val):
                mlflow.log_metric(f"{approach}_{key}", float(val))
    acc_delta = (summary.iloc[1]["mean_accuracy"] or 0) - (summary.iloc[0]["mean_accuracy"] or 0)
    throughput_ratio = ((summary.iloc[1]["docs_per_s"] or 0) /
                        (summary.iloc[0]["docs_per_s"] or 1))
    mlflow.log_metric("delta_accuracy_ray_minus_chain", float(acc_delta))
    mlflow.log_metric("throughput_ratio_ray_over_chain", float(throughput_ratio))

print(f"accuracy delta (ray - chain): {acc_delta:+.3f}")
print(f"throughput ratio (ray / chain): {throughput_ratio:.2f}x")